In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
pip install pandasql


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_excel('/Users/sheebatheodore/Documents/free2move/free2move_data.xlsx')
df.head()

,VEHICLE_ID,MODEL_ID,STARTED_TIME,FINISHED_TIME,CHARGELEVELSTART,CHARGELEVELEND,CHARGED,SERVICERENTAL
0,1d293c750cc5165af5c7266195b1dac0,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-01-15 05:46:16.972,2023-01-15 06:06:14.466,72.0,65.0,0.0,False
1,ca1c90897ce1f12c092fe96acfc8832c,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-01-15 08:12:02.460,2023-01-15 08:17:27.798,NaN,65.0,0.0,True
2,ddf0231bd86205eefbb6fde99b1ab756,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-01-15 10:19:51.668,2023-01-15 10:36:08.477,60.0,46.0,0.0,False
3,981fb8da358abde5fc063be80e011720,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-01-15 12:57:37.334,2023-01-15 12:59:16.264,NaN,100.0,0.0,True
4,981fb8da358abde5fc063be80e011720,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-01-15 13:00:56.392,2023-01-15 13:04:52.894,NaN,98.0,0.0,True


## Exploratory Data Analysis

### Data Overview - Python & SQL

In [4]:
conn = sqlite3.connect(':memory:')
df.to_sql('rentals', conn, index=False, if_exists='replace')

def run_query(sql):
    return pd.read_sql_query(sql, conn)

In [5]:
run_query("""
    SELECT COUNT(*) AS total_rows
    FROM rentals
""")

,total_rows
0,45919


In [6]:
run_query("""
    PRAGMA table_info(rentals)
""")

,cid,name,type,notnull,dflt_value,pk
0,0,VEHICLE_ID,TEXT,0,None,0
1,1,MODEL_ID,TEXT,0,None,0
2,2,STARTED_TIME,TIMESTAMP,0,None,0
3,3,FINISHED_TIME,TIMESTAMP,0,None,0
4,4,CHARGELEVELSTART,REAL,0,None,0
5,5,CHARGELEVELEND,REAL,0,None,0
6,6,CHARGED,REAL,0,None,0
7,7,SERVICERENTAL,INTEGER,0,None,0


In [7]:
df.describe()

,STARTED_TIME,FINISHED_TIME,CHARGELEVELSTART,CHARGELEVELEND,CHARGED
count,45919,45919,40149.000000,45661.000000,45907.000000
mean,2023-04-26 05:36:00.484858112,2023-04-26 06:14:25.533581824,62.680739,55.973807,0.404056
min,2022-12-28 07:09:20.727000,2023-01-01 00:11:48.447000,0.000000,1.000000,0.000000
25%,2023-02-28 08:07:35.136999936,2023-02-28 09:12:02.436499968,42.000000,33.000000,0.000000
50%,2023-04-24 07:44:22.496999936,2023-04-24 08:27:49.974000128,63.000000,55.000000,0.000000
75%,2023-06-21 12:57:18.012999936,2023-06-21 13:56:03.836499968,85.000000,80.000000,1.000000
max,2023-09-17 10:40:53.548000,2023-09-17 10:41:09.011000,100.000000,100.000000,1.000000
std,NaN,NaN,24.288224,27.110900,0.490714


In [16]:
print((df['CHARGELEVELEND'] == 100).sum())
print((df['CHARGELEVELEND'] == 0).sum())
print((df['CHARGELEVELSTART'] == 100).sum())
print((df['CHARGELEVELSTART'] == 0).sum())


1363
0
1550
3


In [12]:
run_query("""
    SELECT 
        CHARGED,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rentals), 1) AS pct
    FROM rentals
    GROUP BY CHARGED
""")

,CHARGED,total,pct
0,NaN,12,0.0
1,0.0,27358,59.6
2,1.0,18549,40.4


In [13]:
run_query("""
    SELECT 
        SERVICERENTAL,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rentals), 1) AS pct
    FROM rentals
    GROUP BY SERVICERENTAL
""")

,SERVICERENTAL,total,pct
0,0,40156,87.4
1,1,5763,12.6


In [17]:
run_query("""
    SELECT
        COUNT(DISTINCT VEHICLE_ID)   AS unique_vehicles,
        COUNT(DISTINCT MODEL_ID)     AS unique_models,
        COUNT(DISTINCT CHARGED)      AS unique_charged,
        COUNT(DISTINCT SERVICERENTAL) AS unique_servicerental
    FROM rentals
""")

,unique_vehicles,unique_models,unique_charged,unique_servicerental
0,52,1,2,2


In [18]:
# date range
run_query("""
    SELECT
        MIN(STARTED_TIME) AS first_rental,
        MAX(STARTED_TIME) AS last_rental
    FROM rentals
""")

,first_rental,last_rental
0,2022-12-28 07:09:20.727000,2023-09-17 10:40:53.548000


### Data Quality

In [19]:
df.isnull().sum() / len(df) * 100

VEHICLE_ID           0.000000
MODEL_ID             0.000000
STARTED_TIME         0.000000
FINISHED_TIME        0.000000
CHARGELEVELSTART    12.565605
CHARGELEVELEND       0.561859
CHARGED              0.026133
SERVICERENTAL        0.000000
dtype: float64

In [20]:
# Missing CHARGELEVELSTART by rental type
df[df['CHARGELEVELSTART'].isnull()]['SERVICERENTAL'].value_counts()

SERVICERENTAL
True     5760
False      10
Name: count, dtype: int64

In [21]:
df[df['CHARGELEVELEND'].isnull()]['SERVICERENTAL'].value_counts()

SERVICERENTAL
False    250
True       8
Name: count, dtype: int64

In [22]:
# Quantify how much of the missing is explained by service rentals
missing_service  = df[df['SERVICERENTAL']==True]['CHARGELEVELSTART'].isnull().sum()
missing_customer = df[df['SERVICERENTAL']==False]['CHARGELEVELSTART'].isnull().sum()
total_missing    = df['CHARGELEVELSTART'].isnull().sum()

print(f'Missing from service rentals  : {missing_service}')
print(f'Missing from customer rentals : {missing_customer}')
print(f'Total missing                 : {total_missing}')
print(f'Explained by service rentals  : {missing_service/total_missing*100:.1f}%')

Missing from service rentals  : 5760
Missing from customer rentals : 10
Total missing                 : 5770
Explained by service rentals  : 99.8%


In [23]:
#check for duplicates
print(df.duplicated().sum())

218


In [24]:
df.duplicated(subset=['VEHICLE_ID', 'STARTED_TIME', 'FINISHED_TIME']).sum()

np.int64(224)

In [25]:
df[df.duplicated(keep=False)].sort_values(['VEHICLE_ID','STARTED_TIME', 'FINISHED_TIME']).head(5)

,VEHICLE_ID,MODEL_ID,STARTED_TIME,FINISHED_TIME,CHARGELEVELSTART,CHARGELEVELEND,CHARGED,SERVICERENTAL
10138,0592a3e4a455ba32227b24f88215cce6,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-05-04 18:24:13.392,2023-05-04 18:28:50.842,94.0,92.0,0.0,False
18154,0592a3e4a455ba32227b24f88215cce6,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-05-04 18:24:13.392,2023-05-04 18:28:50.842,94.0,92.0,0.0,False
18309,0819698514bc1e0ddaaad0522259d4dd,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-05-04 04:41:54.522,2023-05-04 04:56:44.926,39.0,35.0,1.0,False
44377,0819698514bc1e0ddaaad0522259d4dd,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-05-04 04:41:54.522,2023-05-04 04:56:44.926,39.0,35.0,1.0,False
35674,0819698514bc1e0ddaaad0522259d4dd,7d3278bd8d29df4dbb5dae11ed4e69bd,2023-05-04 08:50:36.949,2023-05-04 09:00:16.911,35.0,32.0,0.0,False


In [26]:
#drpop exact duplicates
df_clean = df.drop_duplicates()

print(f'Before : {len(df)} rows')
print(f'After  : {len(df_clean)} rows')
print(f'Dropped: {len(df) - len(df_clean)} rows')

Before : 45919 rows
After  : 45701 rows
Dropped: 218 rows


In [27]:
df_clean.shape

(45701, 8)

In [28]:
conflicting = df_clean[df_clean.duplicated(subset=['VEHICLE_ID','STARTED_TIME','FINISHED_TIME'], keep=False)]
conflicting[['VEHICLE_ID','STARTED_TIME','FINISHED_TIME',
             'CHARGELEVELSTART','CHARGELEVELEND',
             'CHARGED','SERVICERENTAL']].sort_values(['VEHICLE_ID','STARTED_TIME'])

,VEHICLE_ID,STARTED_TIME,FINISHED_TIME,CHARGELEVELSTART,CHARGELEVELEND,CHARGED,SERVICERENTAL
7805,0819698514bc1e0ddaaad0522259d4dd,2023-07-22 19:42:30.280,2023-07-22 19:43:32.063,33.0,33.0,0.0,False
7806,0819698514bc1e0ddaaad0522259d4dd,2023-07-22 19:42:30.280,2023-07-22 19:43:32.063,33.0,33.0,1.0,False
29839,0c5d1b8f3dbe2f559fd3b70a6a31a0ce,2023-05-13 17:42:20.171,2023-05-13 17:51:39.827,31.0,28.0,0.0,False
29840,0c5d1b8f3dbe2f559fd3b70a6a31a0ce,2023-05-13 17:42:20.171,2023-05-13 17:51:39.827,31.0,28.0,1.0,False
33678,1c347a2ffbbdfb70197160e8e77c9a49,2023-07-23 12:13:06.447,2023-07-23 12:15:51.103,30.0,31.0,0.0,False
33679,1c347a2ffbbdfb70197160e8e77c9a49,2023-07-23 12:13:06.447,2023-07-23 12:15:51.103,30.0,31.0,1.0,False
36921,61bedc330d875425f1eb7937a694cc21,2023-07-16 11:23:16.902,2023-07-16 12:07:45.154,28.0,73.0,0.0,False
36922,61bedc330d875425f1eb7937a694cc21,2023-07-16 11:23:16.902,2023-07-16 12:07:45.154,28.0,73.0,1.0,False
11056,6a40f9c6a146ffdfa0a820cde2662af8,2023-07-16 13:10:37.727,2023-07-16 17:50:50.164,49.0,37.0,1.0,False
39976,6a40f9c6a146ffdfa0a820cde2662af8,2023-07-16 13:10:37.727,2023-07-16 17:50:50.164,49.0,37.0,0.0,False


In [29]:

# Identify conflicting rows (same vehicle + timestamps but CHARGED differs)
conflicting_idx = df_clean[df_clean.duplicated(
    subset=['VEHICLE_ID', 'STARTED_TIME', 'FINISHED_TIME'], keep=False
)].index

# Drop conflicting rows
df_clean = df_clean.drop(conflicting_idx)

print(f'Original : {len(df)} rows')
print(f'Cleaned  : {len(df_clean)} rows')
print(f'Dropped  : {len(df) - len(df_clean)} rows')

Original : 45919 rows
Cleaned  : 45689 rows
Dropped  : 230 rows


Note
Data Quality : Duplicates (230 rows removed)

- 218 exact duplicates: identical across all columns, likely a logging/pipeline error. Dropped safely.

- 12 conflicting records (6 pairs): same vehicle and timestamps but disagreeing CHARGED value. Since charge_delta gave no consistent rule to prefer one over the other, and these represent only 0.013% of the data, all 12 were dropped.

In [30]:
# Add duration column
df_clean['duration_min'] = (df_clean['FINISHED_TIME'] - df_clean['STARTED_TIME']).dt.total_seconds() / 60

# 1. Negative durations
print('Negative durations:', (df_clean['duration_min'] < 0).sum())

# 2. Very short rentals (< 1 min)
print('Short rentals < 1 min:', (df_clean['duration_min'] < 1).sum())

# 3. Very long rentals (> 24h)
print('Long rentals > 24h:', (df_clean['duration_min'] > 1440).sum())

# 4. Charge levels out of range
print('CHARGELEVELSTART < 0  :', (df_clean['CHARGELEVELSTART'] < 0).sum())
print('CHARGELEVELSTART > 100:', (df_clean['CHARGELEVELSTART'] > 100).sum())
print('CHARGELEVELEND < 0    :', (df_clean['CHARGELEVELEND'] < 0).sum())
print('CHARGELEVELEND > 100  :', (df_clean['CHARGELEVELEND'] > 100).sum())

Negative durations: 3
Short rentals < 1 min: 811
Long rentals > 24h: 133
CHARGELEVELSTART < 0  : 0
CHARGELEVELSTART > 100: 0
CHARGELEVELEND < 0    : 0
CHARGELEVELEND > 100  : 0


In [31]:
# Drop negative durations
df_clean = df_clean[df_clean['duration_min'] >= 0]

print(f'Original rows : {len(df)}')
print(f'Final cleaned : {len(df_clean)}')
print(f'Total dropped : {len(df) - len(df_clean)}')

Original rows : 45919
Final cleaned : 45686
Total dropped : 233


**3 negative durations removed:**
- 1 due to timestamp recording error
- 2 caused by Daylight Saving Time clock change on 2023-03-26
- Cannot recover real duration → droppe

In [32]:
short = df_clean[df_clean['duration_min'] < 1]

print(f'Total short rentals < 1 min: {len(short)}')

# Count
short['SERVICERENTAL'].value_counts()

# Percentage
short['SERVICERENTAL'].value_counts(normalize=True).mul(100).round(1)

Total short rentals < 1 min: 808


SERVICERENTAL
False    63.0
True     37.0
Name: proportion, dtype: float64

In [ ]:

df_clean['is_short_rental'] = (df_clean['duration_min'] < 1) & (df_clean['SERVICERENTAL'] == False)

print(df_clean['is_short_rental'].sum()) 

509


In [34]:
short = df_clean[df_clean['duration_min'] < 1].copy()
short['duration_sec'] = short['duration_min'] * 60

# Highest and lowest
print(f'Highest: {short["duration_sec"].max():.1f} seconds')
print(f'Lowest : {short["duration_sec"].min():.1f} seconds')

# 5 shortest
short.nsmallest(5, 'duration_sec')[['STARTED_TIME','duration_sec','CHARGELEVELSTART','CHARGELEVELEND','CHARGED','SERVICERENTAL']]

# 5 longest
short.nlargest(5, 'duration_sec')[['STARTED_TIME','duration_sec','CHARGELEVELSTART','CHARGELEVELEND','CHARGED','SERVICERENTAL']]

Highest: 59.8 seconds
Lowest : 1.1 seconds


,STARTED_TIME,duration_sec,CHARGELEVELSTART,CHARGELEVELEND,CHARGED,SERVICERENTAL
9164,2023-03-16 13:49:24.436,59.756,100.0,100.0,0.0,False
7914,2023-07-27 09:58:57.623,59.718,48.0,48.0,0.0,False
41074,2023-02-15 13:46:09.781,59.608,NaN,100.0,0.0,True
2702,2023-01-02 08:48:26.955,59.606,NaN,100.0,1.0,True
14947,2023-03-08 16:48:18.469,59.565,53.0,53.0,0.0,False


NOte: These are almost certainly accidental unlocks or immediate cancellations by customers  no one can drive anywhere in under 60 seconds. Safe to flag and exclude from trip analysis

In [35]:
customer_rentals = df_clean[
    (df_clean['SERVICERENTAL'] == False) & 
    (df_clean['duration_min'] >= 1)
]

In [36]:
long = df_clean[df_clean['duration_min'] > 1440].copy()
long['duration_days'] = (long['duration_min'] / 60 / 24).round(1)

print(f'Total long rentals > 24h: {len(long)}')

# Breakdown
long['SERVICERENTAL'].value_counts()

# Stats in days
long['duration_days'].describe().round(1)

# Top 10 longest
long.nlargest(10, 'duration_min')[['STARTED_TIME','FINISHED_TIME','duration_days',
                                    'CHARGELEVELSTART','CHARGELEVELEND','CHARGED','SERVICERENTAL']]

Total long rentals > 24h: 133


,STARTED_TIME,FINISHED_TIME,duration_days,CHARGELEVELSTART,CHARGELEVELEND,CHARGED,SERVICERENTAL
41631,2023-03-12 15:41:45.173,2023-03-26 15:20:29.011,14.0,30.0,24.0,1.0,False
15022,2023-03-04 15:09:52.473,2023-03-12 15:12:51.819,8.0,48.0,30.0,1.0,False
45897,2023-08-27 21:11:45.704,2023-09-01 19:43:01.873,4.9,87.0,40.0,1.0,False
2685,2022-12-28 07:09:20.727,2023-01-02 05:07:10.824,4.9,72.0,100.0,1.0,False
15657,2023-04-13 15:55:50.231,2023-04-18 12:16:34.055,4.8,100.0,61.0,1.0,False
1708,2023-05-15 11:37:27.390,2023-05-20 03:48:46.035,4.7,89.0,93.0,1.0,False
45915,2023-09-11 05:01:33.619,2023-09-15 11:05:58.017,4.3,28.0,52.0,1.0,False
5423,2023-09-12 05:35:47.510,2023-09-16 06:22:54.319,4.0,81.0,62.0,1.0,False
7467,2023-05-30 06:30:37.753,2023-06-03 06:28:45.705,4.0,62.0,35.0,1.0,False
29193,2023-04-06 15:55:19.664,2023-04-10 14:03:15.415,3.9,64.0,48.0,1.0,False


In [37]:
long.groupby(['SERVICERENTAL', 'CHARGED']).size().reset_index(name='count')

,SERVICERENTAL,CHARGED,count
0,False,0.0,19
1,False,1.0,114


**Note**

Long Rentals > 24h (133 rows)

All 133 are customer rentals: zero service rentals
- 86% (114) were charged during the rental: xpected for multi-day rentals
- Could represent legitimate long rentals, forgotten cars, or unclosed sessions
- they are valid customer events and represent real behaviour worth analysing

In [38]:
df_clean.shape

(45686, 10)

In [39]:
import plotly.express as px
import pandas as pd

# --- prep ---
df_clean['duration_bucket'] = pd.cut(
    df_clean['duration_min'],
    bins=[0, 1, 5, 15, 30, 60, 360, 1440, df_clean['duration_min'].max()],
    labels=['<1 min','1-5 min','5-15 min','15-30 min','30-60 min','1-6h','6-24h','>24h']
)
bucket_counts = df_clean['duration_bucket'].value_counts().sort_index().reset_index()
bucket_counts.columns = ['duration_bucket','count']

In [40]:
# Chart 1 — rental duration distribution
colors = ['#E24B4A' if x == '<1 min' else '#EF9F27' if x == '>24h' 
          else '#378ADD' for x in bucket_counts['duration_bucket']]

fig1 = px.bar(
    bucket_counts,
    x='duration_bucket',
    y='count',
    title='Rental duration distribution',
    labels={'duration_bucket': 'Duration', 'count': 'Number of rentals'},
    template='plotly_white'
)
fig1.update_traces(marker_color=colors)
fig1.show()

In [41]:
# Chart 2 — short rentals breakdown
short = df_clean[df_clean['duration_min'] < 1]
short_counts = short['SERVICERENTAL'].map({True: 'Service agent', False: 'Customer'}) \
                                     .value_counts().reset_index()
short_counts.columns = ['type', 'count']

fig2 = px.bar(
    short_counts,
    x='count',
    y='type',
    orientation='h',
    title='Short rentals (<1 min) — customer vs service',
    labels={'count': 'Number of rentals', 'type': ''},
    color='type',
    color_discrete_map={'Customer': '#378ADD', 'Service agent': '#1D9E75'},
    template='plotly_white'
)
fig2.update_layout(showlegend=False)
fig2.show()

In [42]:
# Chart 3 — long rentals charged breakdown
long = df_clean[df_clean['duration_min'] > 1440]
long_charged = long['CHARGED'].map({1.0: 'Charged', 0.0: 'Not charged'}) \
                               .value_counts().reset_index()
long_charged.columns = ['charged', 'count']

fig3 = px.bar(
    long_charged,
    x='count',
    y='charged',
    orientation='h',
    title='Long rentals (>24h) — charged vs not charged',
    labels={'count': 'Number of rentals', 'charged': ''},
    color='charged',
    color_discrete_map={'Charged': '#1D9E75', 'Not charged': '#E24B4A'},
    template='plotly_white'
)
fig3.update_layout(showlegend=False)
fig3.show()

### Rental Duration Analysis

In [44]:
# All rentals
df_clean['duration_min'].describe().round(2)


count    45686.00
mean        38.44
std        221.22
min          0.02
25%          9.53
50%         15.80
75%         25.02
max      20138.73
Name: duration_min, dtype: float64

In [45]:
fig = px.histogram(
    df_clean[df_clean['duration_min'] <= 200],
    x='duration_min',
    nbins=80,
    title='Rental duration distribution (capped at 200 min)',
    labels={'duration_min': 'Duration (minutes)'},
    template='plotly_white'
)

fig.show()

Notes:
1. Typical rental is very short

- Median rental duration is 16 minutes, mostly is used for quick urban trips, not long journeys

2. Median

- Mean (38 min) is almost double the median (16 min) due to extreme outliers like the 14-day rental. Median is the honest metric here.

3. Right skewed distribution

- Skewness of 32: majority of rentals cluster between 5–25 minutes with a long tail pulling the mean up

4. 75% of rentals are under 25 minutes

- Confirms the spontaneous, short-trip nature of the service 

5. Short rentals < 1 min (808)

- Likely cancelled bookings or accidental unlocks  worth monitoring as it represents friction in the customer experience

6. Long rentals > 24h (133)

- All customer rentals, 86% were charged could be legitimate multi-day use or forgotten cars. Worth investigating operationally.

In [46]:
# Stats by rental type
df_clean.groupby('SERVICERENTAL')['duration_min'].describe().round(2)


,count,mean,std,min,25%,50%,75%,max
SERVICERENTAL,,,,,,,,
False,39945.0,41.20,236.22,0.02,10.3,16.28,25.05,20138.73
True,5741.0,19.22,27.68,0.08,3.1,10.49,24.68,1047.01


In [47]:

# Median by rental type
df_clean.groupby('SERVICERENTAL')['duration_min'].median().round(2)

SERVICERENTAL
False    16.28
True     10.49
Name: duration_min, dtype: float64

### Usage patterns - how customers use free2move

In [48]:
# Extract time features
df_clean['hour']      = df_clean['STARTED_TIME'].dt.hour
df_clean['dayofweek'] = df_clean['STARTED_TIME'].dt.day_name()
df_clean['month']     = df_clean['STARTED_TIME'].dt.month_name()



In [49]:
# Rentals by hour
df_clean.groupby('hour').size()

hour
0      423
1      355
2      429
3      511
4      941
5     1626
6     2274
7     2445
8     2404
9     2308
10    2544
11    2605
12    2604
13    2747
14    2837
15    3004
16    3096
17    3070
18    2485
19    2135
20    1878
21    1344
22     956
23     665
dtype: int64

Note:
Morning commute (8am) is NOT the peak aeound afternoon (16h) is. This suggests Free2Move is used more for leisure and errands than pure commuting

In [50]:
# Rentals by day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df_clean.groupby('dayofweek').size().reindex(day_order)

dayofweek
Monday       6363
Tuesday      6310
Wednesday    6643
Thursday     6688
Friday       7261
Saturday     7159
Sunday       5262
dtype: int64

Note:
 There is a clear weekday plateau → weekend peak → Sunday drop pattern. This is classic leisure-driven usage on top of a steady weekday base.

In [51]:
#Rentls by month
month_order = ['December','January','February','March','April','May','June','July','August','September']
df_clean.groupby('month').size().reindex(month_order)


month
December        6
January      6080
February     5543
March        6534
April        6218
May          5970
June         5807
July         5612
August       3440
September     476
dtype: int64

**Note**

Usage follows a clear seasonal pattern peaks in spring (March), stable through early summer, then collapses in August as suggesting holidays

### Battery analysis

In [ ]:
# customer rentals only: service rentals have no CHARGELEVELSTART
# short rentals (<1 min) retained: impact on battery stats is negligible (0.09% difference)
customer = df_clean[df_clean['SERVICERENTAL']==False].copy()
customer['charge_delta'] = customer['CHARGELEVELEND'] - customer['CHARGELEVELSTART']


In [53]:

# Start level stats
customer['CHARGELEVELSTART'].describe().round(2)


count    39935.00
mean        62.71
std         24.29
min         10.00
25%         42.00
50%         63.00
75%         85.00
max        100.00
Name: CHARGELEVELSTART, dtype: float64

In [54]:

# End level stats
customer['CHARGELEVELEND'].describe().round(2)



count    39695.00
mean        55.72
std         24.61
min          1.00
25%         35.00
50%         55.00
75%         77.00
max        100.00
Name: CHARGELEVELEND, dtype: float64

In [55]:
# Battery consumption
customer['charge_delta'].describe().round(2)



count    39695.00
mean        -6.98
std          8.36
min        -98.00
25%         -9.00
50%         -5.00
75%         -2.00
max         78.00
Name: charge_delta, dtype: float64

In [56]:
# Start level buckets
bins   = [0, 20, 40, 60, 80, 100]
labels = ['0-20%','21-40%','41-60%','61-80%','81-100%']
customer['start_bucket'] = pd.cut(customer['CHARGELEVELSTART'], bins=bins, labels=labels)
customer['start_bucket'].value_counts().sort_index()


start_bucket
0-20%       1010
21-40%      8406
41-60%      9309
61-80%      9114
81-100%    12096
Name: count, dtype: int64

In [57]:

# Cars hitting 100% at end
(customer['CHARGELEVELEND'] == 100).sum()
(customer['CHARGELEVELEND'] == 100).mean() * 100

np.float64(0.9888596820628364)

Note:

**Customer Battery Analysis**

***Charge at Start (CHARGELEVELSTART):***

- Average start level: 62.7% : fleet is reasonably well charged when customers pick up
- 25% of rentals start below 42% : a quarter of customers get a low battery car
- Minimum start level: 10% : some cars are nearly dead when rented
- Maximum: 100% , some customers get a fully charged car

***Charge at End (CHARGELEVELEND):***

- Average end level: 55.7% , drops ~7% per trip on average
- 25% of rentals end below 35% of battery, cars returned fairly low
- Minimum end level: 1% — cars returned almost completely dead

***Battery Consumption (charge_delta):***

- Median consumption: -5% per trip — very small, short trips don't drain much
- Mean consumption: -7% per trip
- Max loss: -98% , someone drained almost a full battery
- Max gain: +78% , customer charged heavily during rental

***Start Level Buckets:***

- Most rentals start well charged — 81-100% bucket has 12,096 rentals (30%)
- But 1,010 rentals (2.5%) start below 20%, operational risk

***Cars hitting 100% at end:***

Only ~1% of customer rentals end at 100% — overstay risk from customers is very low

In [ ]:

customer = df_clean[df_clean['SERVICERENTAL']==False].copy()
customer['charge_delta'] = customer['CHARGELEVELEND'] - customer['CHARGELEVELSTART']



In [60]:
 #All 4 combinations overview
pd.crosstab(df_clean['SERVICERENTAL'], df_clean['CHARGED'], margins=True)



CHARGED,0.0,1.0,All
SERVICERENTAL,,,
False,24502,15433,39935
True,2757,2982,5739
All,27259,18415,45674


In [61]:
# Customer CHARGED=False
c_false = customer[customer['CHARGED']==False]
c_false[['CHARGELEVELSTART','CHARGELEVELEND','charge_delta']].describe().round(2)



,CHARGELEVELSTART,CHARGELEVELEND,charge_delta
count,24502.00,24344.00,24344.00
mean,63.60,56.52,-7.07
std,23.38,23.80,7.40
min,10.00,1.00,-95.00
25%,44.00,37.00,-9.00
50%,65.00,56.00,-5.00
75%,84.00,77.00,-3.00
max,100.00,100.00,10.00


In [62]:
# Customer CHARGED=True
c_true = customer[customer['CHARGED']==True]
c_true[['CHARGELEVELSTART','CHARGELEVELEND','charge_delta']].describe().round(2)

,CHARGELEVELSTART,CHARGELEVELEND,charge_delta
count,15433.00,15351.00,15351.00
mean,61.31,54.43,-6.85
std,25.59,25.80,9.68
min,10.00,1.00,-98.00
25%,38.00,32.00,-9.00
50%,60.00,52.00,-4.00
75%,85.00,78.00,-2.00
max,100.00,100.00,78.00


Note:

Both groups consume roughly the same battery per trip (~5%). The CHARGED=True group shows slightly less net loss because of partial charging, but customers are not significantly recharging the fleet they're just topping up slightly. Agents are still needed to do the heavy charging work.

### Service Rental

In [63]:
service = df_clean[df_clean['SERVICERENTAL']==True].copy()

# Split into two cases
case1 = service[service['CHARGED']==True].copy()   # charging mission
case2 = service[service['CHARGED']==False].copy()  # moving mission

bins   = [0, 20, 40, 60, 80, 100]
labels = ['0-20%','21-40%','41-60%','61-80%','81-100%']

In [64]:
# Case 1
print(f'Case 1 count: {len(case1)}')
case1['CHARGELEVELEND'].describe().round(2)
case1['end_bucket'] = pd.cut(case1['CHARGELEVELEND'], bins=bins, labels=labels)
case1['end_bucket'].value_counts().sort_index()

Case 1 count: 2982


end_bucket
0-20%      1674
21-40%      203
41-60%      127
61-80%      129
81-100%     848
Name: count, dtype: int64

In [65]:
# Case 2
print(f'Case 2 count: {len(case2)}')
case2['CHARGELEVELEND'].describe().round(2)
case2['end_bucket'] = pd.cut(case2['CHARGELEVELEND'], bins=bins, labels=labels)
case2['end_bucket'].value_counts().sort_index()

Case 2 count: 2757


end_bucket
0-20%       406
21-40%       86
41-60%       94
61-80%      209
81-100%    1957
Name: count, dtype: int64

Notes:

**Service Agent Battery Analysis**
Since CHARGELEVELSTART is missing for 99.9% of service rentals, only CHARGELEVELEND can be analysed. This tells us what battery level the car is at when the agent ends their rental.


***Case 1 — Charging Mission (CHARGED=True, 2,982 rentals)***

- Agent picks up a dead car, drives it to a charger, plugs it in and ends rental immediately
- Median end level: 16%, car is still low when agent leaves, continues charging on its own
- 1,674 rentals (56%) end below 20%, confirms agent drops off and leaves early
- 848 rentals (28%) end above 80%, some agents wait longer before ending rental
- High std (38%) shows inconsistent agent behaviour, no standard process for when to end the rental



***Case 2 — Moving Mission (CHARGED=False, 2,757 rentals)***

- Agent moves a fully charged car away from charger to avoid overstay fees
- Median end level: 94%, cars are well charged when moved 
- 1,957 rentals (71%) end above 80%, agents correctly moving high charge cars
- 406 rentals (15%) end below 20%, suspicious, why is an agent moving a nearly dead car away from a charger? Possible operational error worth investigating



***Key insight:***
The gap between Case 1 (median 16%) and Case 2 (median 94%) perfectly describes the two-sided operation agents rescue dead cars AND relocate full ones. CHARGELEVELEND is the only window into service operations and it tells a very clean story.

### Question 2: BATTERY THRESHOLD ANALYSIS 

> Question: At what battery level should we send an agent to move a charging car?

In [ ]:
# STEP 1 — Sort by vehicle and time
df_sorted = df_clean.sort_values(
    ['VEHICLE_ID', 'STARTED_TIME']
).reset_index(drop=True)

# STEP 2 — Get next rental details for each row
df_sorted['next_SERVICERENTAL']    = df_sorted.groupby('VEHICLE_ID')['SERVICERENTAL'].shift(-1)
df_sorted['next_CHARGED']          = df_sorted.groupby('VEHICLE_ID')['CHARGED'].shift(-1)
df_sorted['next_CHARGELEVELSTART'] = df_sorted.groupby('VEHICLE_ID')['CHARGELEVELSTART'].shift(-1)
df_sorted['next_CHARGELEVELEND']   = df_sorted.groupby('VEHICLE_ID')['CHARGELEVELEND'].shift(-1)


In [ ]:
# STEP 3 — Filter to charging missions only
charging_missions = df_sorted[
    (df_sorted['SERVICERENTAL'] == True) &
    (df_sorted['CHARGED'] == True)
].copy()

In [ ]:
# STEP 4 — Classify what happened next
charging_missions['next_event'] = 'other'
charging_missions.loc[
    charging_missions['next_SERVICERENTAL'] == False,
    'next_event'] = 'customer_rented'
charging_missions.loc[
    (charging_missions['next_SERVICERENTAL'] == True) &
    (charging_missions['next_CHARGED'] == False),
    'next_event'] = 'agent_moved'

In [ ]:
# STEP 5 — Focus on customer vs agent only
relevant = charging_missions[
    charging_missions['next_event'].isin(['customer_rented', 'agent_moved'])
].copy()


In [ ]:
# STEP 6 — Assign decision point battery level
# customer → next_CHARGELEVELSTART (exact)
# agent    → next_CHARGELEVELEND   (best estimate — short trip, minimal battery change)
relevant['decision_battery'] = np.where(
    relevant['next_event'] == 'customer_rented',
    relevant['next_CHARGELEVELSTART'],
    relevant['next_CHARGELEVELEND']
)
relevant = relevant.dropna(subset=['decision_battery'])

In [ ]:
# STEP 7 — Full picture with 10% buckets
bins_full   = list(range(0, 110, 10))
labels_full = [f'{i}-{i+10}%' for i in range(0, 100, 10)]
relevant['decision_bucket_full'] = pd.cut(
    relevant['decision_battery'], bins=bins_full, labels=labels_full
)
summary_full = relevant.groupby(
    ['decision_bucket_full', 'next_event']
).size().unstack(fill_value=0)
summary_full['total']                  = summary_full.sum(axis=1)
summary_full['customer_pickup_rate_%'] = (summary_full['customer_rented'] / summary_full['total'] * 100).round(1)
summary_full['agent_move_rate_%']      = (summary_full['agent_moved'] / summary_full['total'] * 100).round(1)
summary_full[['customer_rented','agent_moved','total',
              'customer_pickup_rate_%','agent_move_rate_%']]

In [ ]:
# STEP 8 — Focused on 80-100% with 5% buckets
high = relevant[relevant['decision_battery'] >= 80].copy()
bins_fine   = [80, 85, 90, 95, 100]
labels_fine = ['80-85%','85-90%','90-95%','95-100%']
high['decision_bucket'] = pd.cut(
    high['decision_battery'], bins=bins_fine, labels=labels_fine, include_lowest=True
)
summary_fine = high.groupby(
    ['decision_bucket', 'next_event']
).size().unstack(fill_value=0)
summary_fine['total']                  = summary_fine.sum(axis=1)
summary_fine['customer_pickup_rate_%'] = (summary_fine['customer_rented'] / summary_fine['total'] * 100).round(1)
summary_fine['agent_move_rate_%']      = (summary_fine['agent_moved'] / summary_fine['total'] * 100).round(1)
summary_fine[['customer_rented','agent_moved','total',
              'customer_pickup_rate_%','agent_move_rate_%']]

## Note Intepretation

### Approach
For each charging mission (agent plugs car in), we tracked what happened next:
- **Customer rented** → used `next_CHARGELEVELSTART` (exact battery at pickup)
- **Agent moved** → used `next_CHARGELEVELEND` (best estimate: short trips, minimal battery change)

### Step 1 — Full picture (10% buckets)

| Battery | Customer% | Agent% | Total |
|---|---|---|---|
| 0-10% | 0.0% | 100.0% | 17 |
| 10-20% | 25.0% | 75.0% | 16 |
| 20-80% | 56-80% | 20-44% | low data |
| 80-90% | 54.9% | 45.1% | 257 |
| 90-100% | 60.8% | 39.2% | 1,899 |

>  79% of all events happen in the 90-100% bucket,  most decisions happen at high battery levels

### Step 2 — Focused on 80-100% (5% buckets)

| Battery | Customer% | Agent% | Total |
|---|---|---|---|
| 80-85% | 61.0% | 39.0% | 105 |
| **85-90%** | **51.6%** | **48.4%** | **159** |
| 90-95% | 58.5% | 41.5% | 390 |
| 95-100% | 61.4% | 38.6% | 1,509 |

### Key Finding
> **85-90% is the critical tipping point**, the only bucket where agent moves 
> nearly equal customer rentals (48.4% vs 51.6%)

### Recommendation
> **Send agent when battery reaches 85%**
> - Below 85% → customers naturally rent the car (60-80% pickup rate)
> - At 85-90% → almost 50/50 split, risk of overstay fees rises sharply
> - Waiting beyond 90% → 39% of the time an agent is still needed anyway
> 
> Acting at **85%** gives the best balance between:
> - Avoiding unnecessary agent dispatches (car might still be rented)
> - Preventing overstay fees (car is close enough to 100% to be a risk)

***Question 1:  What can you say about operations?***
Usage Patterns:

Typical rental is 16 minutes, spontaneous short urban trips
Peak hour is 15-17h — afternoon, not morning commute
Busiest days are Friday & Saturday
August drop, Paris empties for holidays
808 cancelled bookings, customer experience friction

Battery & Charging:

Average battery consumption is only 7% per trip, short trips don't drain much
38.6% of customer rentals involve some charging, customers help maintain fleet
Two types of service rentals, charging mission (median end 16%) and moving mission (median end 94%)
583 repeat charging missions (19.6%) , operational inefficiency

Fleet:

52 vehicles, 1 model
12.6% of rentals are service agents, significant operational cost
One vehicle with only 179 rentals vs average 879, possible underperformer

***Question 2:  At what battery level should we send an agent?***

Send agent when battery reaches 85%

Below 85% → customers naturally rent charging cars (60-80% pickup rate)
At 85-90% → almost 50/50 split → risk of overstay fees rises sharply
Waiting beyond 90% → agents still needed 39% of the time anyway

### Fleet Utilisation analysis

In [66]:
fleet = df_clean.groupby('VEHICLE_ID').agg(
    total_rentals     = ('VEHICLE_ID', 'count'),
    customer_rentals  = ('SERVICERENTAL', lambda x: (x==False).sum()),
    service_rentals   = ('SERVICERENTAL', lambda x: (x==True).sum()),
    avg_battery_start = ('CHARGELEVELSTART', 'mean'),
    avg_battery_end   = ('CHARGELEVELEND', 'mean'),
    avg_duration_min  = ('duration_min', 'mean'),
    total_charged     = ('CHARGED', lambda x: (x==True).sum())
).round(2)

fleet['service_ratio_%'] = (fleet['service_rentals'] / fleet['total_rentals'] * 100).round(1)

# summary
fleet['total_rentals'].describe().round(2)



count      52.00
mean      878.58
std       153.08
min       179.00
25%       807.50
50%       881.00
75%       995.25
max      1093.00
Name: total_rentals, dtype: float64

In [67]:
# top & bottom 10
fleet.sort_values('total_rentals', ascending=False).head(10)

,total_rentals,customer_rentals,service_rentals,avg_battery_start,avg_battery_end,avg_duration_min,total_charged,service_ratio_%
VEHICLE_ID,,,,,,,,
99a39a0fcec88de0c6c067adb64a3bb6,1093,950,143,62.10,56.07,54.78,469,13.1
12700f3e43ce275c51956087a0f60e7e,1084,963,121,61.93,54.62,31.93,470,11.2
0ef2ca2d596258de2677980732d313e3,1070,932,138,62.85,55.57,38.29,461,12.9
42ac25c2d95a476f0186fa454c3def84,1059,917,142,64.00,57.37,33.38,396,13.4
2db363ebf1e6be2201eb12230dc83b41,1055,913,142,62.21,55.19,29.49,452,13.5
ddf0231bd86205eefbb6fde99b1ab756,1051,902,149,61.34,55.59,36.14,427,14.2
af77c8463ac27b3776ae4528d58704dc,1044,918,126,64.18,57.51,36.20,468,12.1
e60b281a0143cacff6cb7abf453d8dc3,1042,908,134,62.74,55.48,35.16,438,12.9
ca1c90897ce1f12c092fe96acfc8832c,1038,908,130,61.53,55.10,39.51,403,12.5


In [68]:
fleet.sort_values('total_rentals', ascending=True).head(10)

,total_rentals,customer_rentals,service_rentals,avg_battery_start,avg_battery_end,avg_duration_min,total_charged,service_ratio_%
VEHICLE_ID,,,,,,,,
dc753ee97d00780d5b36379e37ee78bf,179,148,31,65.82,56.37,25.40,24,17.3
1a74a25866a91c2036c39351e7dc8c2d,607,532,75,62.16,55.45,39.55,243,12.4
981fb8da358abde5fc063be80e011720,696,595,101,62.32,55.90,48.71,314,14.5
d7777da68b50ace0b909872d611db978,709,618,91,63.05,57.20,33.98,248,12.8
18a86bed65d052f506e5dedcda224325,726,630,96,63.61,56.43,30.81,277,13.2
7218b9c79dc23f8ca0aaff4fd88147db,742,661,81,63.13,57.21,43.21,309,10.9
b0021abffae86782e17f773e74fbe4a3,749,661,88,61.97,55.24,35.30,288,11.7
1ad57f261423114ec397fc6bcf156335,756,662,94,61.86,54.92,44.05,263,12.4
80ccf8ddbd6c8eb8a6dd943f7837fb80,766,674,92,63.27,56.14,24.84,254,12.0


In [73]:
# service ratio distribution
fleet['service_ratio_%'].describe().round(2)

count    52.00
mean     12.64
std       1.37
min       9.80
25%      11.90
50%      12.60
75%      13.42
max      17.30
Name: service_ratio_%, dtype: float64

***Fleet Utilization***

| Metric | Value |
|---|---|
| Total vehicles | 52 |
| Average rentals per vehicle | 879 |
| Most used vehicle | 1,093 rentals |
| Least used vehicle | **179 rentals**  |
| Average service ratio | 12.6% |

**Key insights:**
- Fleet is **evenly utilized**:  service ratio is consistent across all vehicles (std 1.37%)
- No vehicle exceeds **20% service ratio** : agent workload is well distributed
- Top 10 vehicles all have **1,038–1,093 rentals**:  very consistent high performers


> Investigate `dc753ee9` - not used after jan, Feb 2023 


In [70]:
# active fleet only (exclude dc753ee9)
active_fleet = fleet[fleet['total_rentals'] > 200]

print(f'Active vehicles     : {len(active_fleet)}')
print(f'Avg rentals/vehicle : {active_fleet["total_rentals"].mean().round(0)}')
print(f'Min rentals         : {active_fleet["total_rentals"].min()}')
print(f'Max rentals         : {active_fleet["total_rentals"].max()}')

Active vehicles     : 51
Avg rentals/vehicle : 892.0
Min rentals         : 607
Max rentals         : 1093


***Fleet Utilization: Excluding Inactive Vehicle***

- **dc753ee9 excluded** : only active Jan–Feb 2023 (likely mechanical failure or retirement)
- **51 active vehicles** remaining
- True average: **895 rentals per active vehicle**
- All 51 active vehicles show consistent 7-9 months of activity
  matching the dataset timeframe

### Task 1: Additional Exploration; Service Agent Intervention by Hour & Day

In [71]:
df_clean['hour']      = df_clean['STARTED_TIME'].dt.hour
df_clean['dayofweek'] = df_clean['STARTED_TIME'].dt.day_name()

service  = df_clean[df_clean['SERVICERENTAL']==True]
customer = df_clean[df_clean['SERVICERENTAL']==False]

# by hour
hourly = pd.concat([
    customer.groupby('hour').size().rename('customer'),
    service.groupby('hour').size().rename('service')
], axis=1)
hourly['service_ratio_%'] = (hourly['service'] / hourly['customer'] * 100).round(1)

# by day
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = pd.concat([
    customer.groupby('dayofweek').size().rename('customer').reindex(day_order),
    service.groupby('dayofweek').size().rename('service').reindex(day_order)
], axis=1)
daily['service_ratio_%'] = (daily['service'] / daily['customer'] * 100).round(1)

# display results
print('=== BY HOUR ===')
print(hourly)




=== BY HOUR ===
      customer  service  service_ratio_%
hour                                    
0          351       72             20.5
1          269       86             32.0
2          336       93             27.7
3          398      113             28.4
4          813      128             15.7
5         1380      246             17.8
6         1910      364             19.1
7         1905      540             28.3
8         1780      624             35.1
9         1721      587             34.1
10        1964      580             29.5
11        2073      532             25.7
12        2158      446             20.7
13        2398      349             14.6
14        2539      298             11.7
15        2825      179              6.3
16        3000       96              3.2
17        3034       36              1.2
18        2450       35              1.4
19        2081       54              2.6
20        1809       69              3.8
21        1283       61              4.8


In [72]:

print('=== BY DAY ===')
print(daily)

=== BY DAY ===
           customer  service  service_ratio_%
dayofweek                                    
Monday         5449      914             16.8
Tuesday        5470      840             15.4
Wednesday      5782      861             14.9
Thursday       5945      743             12.5
Friday         6305      956             15.2
Saturday       6410      749             11.7
Sunday         4584      678             14.8


### Task 1 Additional Exploration : Battery Level When Customers Return Cars

In [74]:
customer = df_clean[df_clean['SERVICERENTAL']==False].copy()

customer['CHARGELEVELEND'].describe().round(2)

bins   = [0, 20, 40, 60, 80, 100]
labels = ['0-20%','21-40%','41-60%','61-80%','81-100%']
customer['end_bucket'] = pd.cut(customer['CHARGELEVELEND'], bins=bins, labels=labels)
customer['end_bucket'].value_counts().sort_index()




end_bucket
0-20%      3006
21-40%     9988
41-60%     9126
61-80%     9296
81-100%    8279
Name: count, dtype: int64

In [75]:
print((customer['CHARGELEVELEND'] < 20).sum())

2648


In [76]:
print((customer['CHARGELEVELEND'] < 10).sum())

405


### Note 

7.6% of cars returned below 20% , these immediately need an agent to charge them. With 45,686 rentals that's ~3,000 urgent charging missions per year. Average battery at return is 55% ,  healthy overall but the low tail is concerning.

### Task 2: Addtional exploration: Time to Charge from X% to 85%

In [77]:
# STEP 1 — sort by vehicle and time
df_sorted = df_clean.sort_values(
    ['VEHICLE_ID', 'STARTED_TIME']
).reset_index(drop=True)

# STEP 2 — get next rental details INCLUDING next_STARTED_TIME
df_sorted['next_SERVICERENTAL']    = df_sorted.groupby('VEHICLE_ID')['SERVICERENTAL'].shift(-1)
df_sorted['next_CHARGED']          = df_sorted.groupby('VEHICLE_ID')['CHARGED'].shift(-1)
df_sorted['next_CHARGELEVELSTART'] = df_sorted.groupby('VEHICLE_ID')['CHARGELEVELSTART'].shift(-1)
df_sorted['next_CHARGELEVELEND']   = df_sorted.groupby('VEHICLE_ID')['CHARGELEVELEND'].shift(-1)
df_sorted['next_STARTED_TIME']     = df_sorted.groupby('VEHICLE_ID')['STARTED_TIME'].shift(-1)  # ← this was missing!

# STEP 3 — filter to charging missions
charging_missions = df_sorted[
    (df_sorted['SERVICERENTAL']==True) &
    (df_sorted['CHARGED']==True)
].copy()

# STEP 4 — classify next event
charging_missions['next_event'] = 'other'
charging_missions.loc[charging_missions['next_SERVICERENTAL']==False, 'next_event'] = 'customer_rented'
charging_missions.loc[
    (charging_missions['next_SERVICERENTAL']==True) &
    (charging_missions['next_CHARGED']==False), 'next_event'] = 'agent_moved'

# STEP 5 — relevant only
relevant = charging_missions[
    charging_missions['next_event'].isin(['customer_rented','agent_moved'])
].copy()

# STEP 6 — decision battery
relevant['decision_battery'] = np.where(
    relevant['next_event'] == 'customer_rented',
    relevant['next_CHARGELEVELSTART'],
    relevant['next_CHARGELEVELEND']
)

# STEP 7 — charging rate
relevant['time_to_next_hrs'] = (
    relevant['next_STARTED_TIME'] - relevant['FINISHED_TIME']
).dt.total_seconds() / 3600

relevant['battery_gained']       = relevant['decision_battery'] - relevant['CHARGELEVELEND']
relevant['charge_rate_%_per_hr'] = relevant['battery_gained'] / relevant['time_to_next_hrs']

# STEP 8 — clean outliers
relevant_clean = relevant[
    (relevant['time_to_next_hrs'] > 0) &
    (relevant['battery_gained'] > 0) &
    (relevant['time_to_next_hrs'] < 24)
].copy()

# STEP 9 — results
median_rate = relevant_clean['charge_rate_%_per_hr'].median()
print(relevant_clean['charge_rate_%_per_hr'].describe().round(2))
print(f'\nMedian charging rate: {median_rate:.1f}% per hour')
print()

# STEP 10 — time to reach 85%
print('=== TIME TO REACH 85% ===')
for start in [10, 20, 30, 40, 50, 60, 70]:
    hours = (85 - start) / median_rate
    print(f'From {start}% → 85%: {hours:.1f} hrs ({hours*60:.0f} mins)')

count    1406.00
mean       37.49
std        28.58
min         0.05
25%        17.81
50%        33.40
75%        52.69
max       502.02
Name: charge_rate_%_per_hr, dtype: float64

Median charging rate: 33.4% per hour

=== TIME TO REACH 85% ===
From 10% → 85%: 2.2 hrs (135 mins)
From 20% → 85%: 1.9 hrs (117 mins)
From 30% → 85%: 1.6 hrs (99 mins)
From 40% → 85%: 1.3 hrs (81 mins)
From 50% → 85%: 1.0 hrs (63 mins)
From 60% → 85%: 0.7 hrs (45 mins)
From 70% → 85%: 0.4 hrs (27 mins)


***Intepretation of Additional Operational Analysis for Q1 and Q2***

#### 1. Service Agent Intervention Timing
- Agents most active **7-10am** (service ratio 28-35%)
- Almost **no agent activity 15-18h** (ratio 1-6%), peak customer hours!
- **Saturday has lowest service ratio (11.7%)** despite being busiest day
-  **Critical misalignment**, agents work mornings, customers peak afternoons

**Recommendation:** Shift agent schedules to cover **13-17h** peak window
and increase weekend coverage, especially Saturday



#### 2. Battery Level at Return
- Average return level: **55%**, healthy overall
- **7.6% of cars (3,006) returned below 20%** , need immediate charging
- **1% (405 cars) returned below 10%**, critically low, risk of stranding next customer
- 25% of cars returned below **35%**, need charging before next rental

**Recommendation:** Alert agents when car is returned below **20%** for
priority charging dispatch



#### 4. Charging Rate & Time to Threshold
- Median charging rate: **33.4% per hour**
- A car at 60% takes **~45 minutes** to reach the 85% threshold
- A car at 70% takes only **~27 minutes** to reach threshold

**Recommendation:** When an agent plugs in a car, the system should
automatically **schedule a move alert** based on current battery level:
- Plugged in at 60% → alert in 45 min
- Plugged in at 70% → alert in 27 min
- This prevents overstay fees without unnecessary agent dispatches


# Final Conclusions

## Task 1: Operational Insights & Recommendations

### Dataset Summary
- **45,686 rentals** | **52 vehicles** | **Jan–Sep 2023**
- **87.4% customer rentals** | **12.6% service agent rentals**


### Key Findings

**1. Usage Patterns**
- Typical rental lasts only **16 minutes**, spontaneous short urban trips
- Peak demand at **16h**: afternoon, not morning commute
- Busiest day: **Friday**, quietest: **Sunday**
- **August drops 47%** vs March peak: Paris empties for holidays
- **808 cancelled bookings (<1 min)** : customer experience friction

**2. Battery & Charging**
- Average battery at pickup: **62.7%** | at return: **55.7%**
- Average consumption: only **7% per trip** : short trips drain little
- **6.6% of cars (2,648) returned below 20%** : need immediate charging
- **38.6% of customers plug in** during rental:  helping maintain fleet charge
- **583 repeat charging missions (19.6%)** : agents returning to same car twice

**3. Agent Operations**
- Agents most active **7-10am** (service ratio 28-35%)
- Almost **no agents 15-18h** (ratio 1-6%) : exactly when customers peak!
- **Saturday lowest agent coverage (11.7%)** despite being 2nd busiest day
- One vehicle (`dc753ee9`) inactive after Feb 2023 : likely mechanical failure



### Recommendations to Management

| # | Recommendation | Why |
|---|---|---|
| 1 | **Shift agent schedules to 13-17h** | Agents are absent during peak customer hours fleet availability suffers |
| 2 | **Increase weekend agent coverage** | Saturday is 2nd busiest day but has lowest agent ratio (11.7%) |
| 3 | **Alert system for cars returned below 20%** | 2,648 urgent charging needs per year prioritise dispatch |
| 4 | **Investigate 583 repeat charging missions** | Same car charged twice = wasted agent trips, fix charger reliability |
| 5 | **Use August for fleet maintenance** | 47% drop in demand = ideal window for servicing vehicles |
| 6 | **Reduce cancelled bookings friction** | 808 accidental unlocks suggest UX improvement needed in app |


##  Task 2:  Battery Threshold Recommendation

### The Question
> *"At what battery level is the chance of a customer renting a charging car too low,  so we should send an agent instead?"*

### Methodology
Using **vehicle timeline analysis** (sequential rental analysis):
1. Identified all **2,982 charging missions** (agent plugs car in)
2. Tracked what happened next for each vehicle:
   - **Customer rented** → used `next_CHARGELEVELSTART` (exact battery at pickup)
   - **Agent moved** → used `next_CHARGELEVELEND` (best estimate short trips, minimal battery change)
3. Calculated customer pickup rate vs agent move rate at each battery level
4. Focused on **80-100% range** with 5% buckets (79% of all events happen here)

### Results

| Battery Level | Customer% | Agent% | Total Events |
|---|---|---|---|
| 80-85% | 61.0% | 39.0% | 105 |
| **85-90%** | **51.6%** | **48.4%** | **159** |
| 90-95% | 58.5% | 41.5% | 390 |
| 95-100% | 61.4% | 38.6% | 1,509 |

###  Recommendation
> **Send agent when battery reaches 85%**

**Why 85%?**
- It is the **only bucket where agent moves nearly equal customer rentals (48.4%)**, the tipping point
- Below 85% → customers naturally rent the car (60-80% pickup rate) → no agent needed
- Waiting beyond 90% → agents still needed 39% of the time anyway → too late
- At **33.4% charge per hour**, a car plugged in at 60% hits 85% in **~45 minutes**  giving enough lead time to dispatch an agent

###  Actionable Rule for Operations
> When an agent plugs in a car, the system should **automatically schedule a move alert**
> based on current battery level:
>
> | Plugged in at | Time until 85% | Action |
> |---|---|---|
> | 40% | ~81 min | Schedule alert in 70 min |
> | 50% | ~63 min | Schedule alert in 52 min |
> | 60% | ~45 min | Schedule alert in 34 min |
> | 70% | ~27 min | Dispatch agent immediately |


###  Limitations
- `CHARGELEVELSTART` missing for all service rentals  agent pickup battery estimated via `CHARGELEVELEND`
- Charging rate varies widely (std 28.6%) median used as best estimate
- No location data — cannot identify which areas have highest overstay risk
- Dataset covers only 9 months seasonal patterns may differ year to year